<a href="https://colab.research.google.com/github/Churdlez/BUS118s-ML_Basics/blob/main/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Data source: Synthetic educational housing dataset generated for this assignment.
# The dataset contains 240 realistic housing records.
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_FILE = Path("house_prices.csv")
df = pd.read_csv(DATA_FILE)

# Features and target
features = [
    "square_footage",
    "bedrooms",
    "home_age_years",
    "location"
]

target = "price"

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Numerical and categorical features
numeric_features = [
    "square_footage",
    "bedrooms",
    "home_age_years"
]

categorical_features = ["location"]

# Encode the location column
preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# Create the model pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
predictions = model.predict(X_test)

print(f"Number of records: {len(df)}")
print(
    f"Test Mean Absolute Error: "
    f"${mean_absolute_error(y_test, predictions):,.0f}"
)
print(f"Test R-squared: {r2_score(y_test, predictions):.3f}")

# Predict price for a new 2,000-square-foot Downtown house
new_house = pd.DataFrame({
    "square_footage": [2000],
    "bedrooms": [3],
    "home_age_years": [10],
    "location": ["Downtown"]
})

predicted_price = model.predict(new_house)[0]

print(
    f"\nPredicted price for a 2,000 sq ft "
    f"Downtown house: ${predicted_price:,.2f}"
)

# Display model coefficients
location_names = (
    model.named_steps["preprocessor"]
    .named_transformers_["location"]
    .get_feature_names_out(categorical_features)
)

feature_names = list(location_names) + [
    "square_footage",
    "bedrooms",
    "home_age_years"
]

coefficients = model.named_steps["regressor"].coef_

print("\nModel Coefficients:")

for feature, coefficient in zip(feature_names, coefficients):
    print(f"{feature}: ${coefficient:,.2f}")

print("\nInterpretation:")
print(
    "The square footage coefficient estimates the average change "
    "in house price for one additional square foot."
)
print(
    "The location coefficients compare Rural and Suburb homes "
    "with Downtown homes, which is the reference location."
)

Number of records: 240
Test Mean Absolute Error: $20,662
Test R-squared: 0.978

Predicted price for a 2,000 sq ft Downtown house: $706,838.06

Model Coefficients:
location_Rural: $-147,545.54
location_Suburb: $-79,718.59
square_footage: $201.75
bedrooms: $19,014.93
home_age_years: $-1,409.87

Interpretation:
The square footage coefficient estimates the average change in house price for one additional square foot.
The location coefficients compare Rural and Suburb homes with Downtown homes, which is the reference location.
